# Experiment Design: Order Review Step on the Direct Checkout Path

## Why an experiment is needed

The funnel analysis found that users reaching checkout via the cart convert at
51.5%, against 37.1% for users who go directly to checkout — a 14.5pp gap
(95% CI [12.5, 16.4], p < 0.001, Cohen's h = 0.29). The gap held across all
seven subgroups tested, spanning three time periods and four traffic channels.

That establishes the gap is real. It does not establish what causes it.

Adding an item to a cart is itself an act of intent. So the gap may reflect
**who chooses each path** rather than **how each path is built**. No amount of
observational subgroup analysis can separate these two explanations — and the
gift-season pattern (the direct path carried 47% of checkout traffic during
gift season and 24% afterwards, while cart-path conversion rose as its share
fell) actively supports the selection explanation.

Random assignment breaks the link between intent and path. That is the only way
to isolate the path's own contribution.

## What is being tested

The naive design — randomly assign users to the cart or direct path — is
impossible. Nobody can be forced to add an item to a cart.

What can be randomised is an **intervention**. If the cart page causes better
conversion, some property of it must be doing the work. The most plausible
candidate is that the cart gives users a moment to review what they are buying
before committing.

**Hypothesis:** Adding an order-review step before payment on the direct
checkout path will increase checkout-to-purchase conversion for those users.

**Primary metric:** checkout → purchase conversion among direct-path users.
Fixed in advance; choosing a metric after seeing results guarantees finding
something that moved by chance.

**Guardrail metric:** total purchase volume across *both* paths.

An earlier draft used median time from `begin_checkout` to `purchase`. That metric
is only computable for users who purchased, so it is conditioned on the outcome
the test is trying to move. If the review step deters low-intent users who would
have bought slowly, the surviving purchasers are faster on average and the
guardrail *improves* while customers are being lost. A guardrail conditioned on
conversion cannot guard against conversion damage.

Total purchase volume is computed on everyone and catches the failure mode that
actually matters commercially: direct-path conversion rising while total orders do
not, meaning users were shifted between paths rather than gained.

**Secondary guardrail:** checkout abandonment rate — the share of users who reach
`begin_checkout` but never `purchase`, across both arms. Also computed on the full
population, and directly sensitive to added friction.

**Randomisation unit:** user, not session. A user seeing the review step on one
visit and not the next would contaminate both arms.

In [1]:
from statsmodels.stats.power import zt_ind_solve_power
from statsmodels.stats.proportion import proportion_effectsize
import math

baseline_rate = 0.3706      # direct-path checkout-to-purchase, from Query 5
mde_pp = 0.03               # minimum detectable effect: 3 percentage points
target_rate = baseline_rate + mde_pp

alpha = 0.05                # false positive rate we accept
power = 0.80                # probability of detecting a true effect of MDE size

print("Baseline:", round(baseline_rate * 100, 2), "%")
print("Target:  ", round(target_rate * 100, 2), "%")
print("Relative lift:", round(mde_pp / baseline_rate * 100, 1), "%")

Baseline: 37.06 %
Target:   40.06 %
Relative lift: 8.1 %


## Choosing the minimum detectable effect

The observed gap is 14.5pp, but most of that is probably selection rather than
path design. Powering the test to detect 14.5pp would be naive — a real 3pp
improvement would go undetected and be wrongly read as "the intervention does
not work."

Too small an MDE has the opposite cost: sample size explodes and the test runs
for months.

**Starting point: 3pp** — roughly a fifth of the observed gap. Conservative
about how much of the gap is causal, while still commercially meaningful
(37.1% → 40.1% is an 8% relative lift).

α = 0.05 (tolerated false-positive rate), power = 0.80 (probability of
detecting a true effect of MDE size). Effect size, α, power and sample size are
mathematically linked — fix any three and the fourth is determined.

Two-sided test: a one-sided test needs fewer users but would be blind to the
intervention making things worse.

In [2]:
effect_size = proportion_effectsize(target_rate, baseline_rate)

n_per_arm = zt_ind_solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative='two-sided'
)

n_per_arm = math.ceil(n_per_arm)

print("Effect size (Cohen's h):", round(effect_size, 4))
print("Users needed per arm:", n_per_arm)
print("Total users needed:", n_per_arm * 2)

Effect size (Cohen's h): 0.0616
Users needed per arm: 4131
Total users needed: 8262


## From sample size to duration

Sample size alone is not actionable. A stakeholder needs weeks.

Duration is rounded up to whole weeks because the Day 1 profiling found a clear
weekly cycle — weekends are consistently quieter. A test running 10 days would
contain two Mondays but one Saturday, biasing the weekday mix between the start
and end of the test.

In [3]:
# Direct-path users reaching checkout: 4,058 over 92 days
direct_checkout_users = 4058
days_observed = 92

daily_direct_users = direct_checkout_users / days_observed
total_needed = n_per_arm * 2
days_required = total_needed / daily_direct_users
weeks_required = days_required / 7

print("Direct-path checkout users per day:", round(daily_direct_users, 1))
print("Days required:", round(days_required, 1))
print("Weeks required:", round(weeks_required, 1))
print("Rounded up to whole weeks:", math.ceil(weeks_required))

Direct-path checkout users per day: 44.1
Days required: 187.3
Weeks required: 26.8
Rounded up to whole weeks: 27


## The duration problem

At a 3pp MDE on direct-path users alone, the test requires 27 weeks — just over
six months. That is not a viable test: the site would change underneath it,
seasonality would confound it, and no business would wait.

Rather than quietly adjusting parameters until the answer looks acceptable, the
table below shows what each level of sensitivity actually costs, for two
candidate populations.

In [4]:
mde_options = [0.02, 0.03, 0.04, 0.05, 0.06, 0.08]

print("MDE (pp) | n per arm | weeks (direct only) | weeks (all checkout)")
print("-" * 65)

for mde in mde_options:
    target = baseline_rate + mde
    es = proportion_effectsize(target, baseline_rate)
    n = math.ceil(zt_ind_solve_power(effect_size=es, alpha=alpha, power=power,
                                     ratio=1.0, alternative='two-sided'))
    weeks_direct = math.ceil((n * 2) / daily_direct_users / 7)
    weeks_all = math.ceil((n * 2) / (9715 / 92) / 7)
    print(str(round(mde*100, 1)).rjust(8), "|",
          str(n).rjust(9), "|",
          str(weeks_direct).rjust(19), "|",
          str(weeks_all).rjust(20))

MDE (pp) | n per arm | weeks (direct only) | weeks (all checkout)
-----------------------------------------------------------------
     2.0 |      9251 |                  60 |                   26
     3.0 |      4131 |                  27 |                   12
     4.0 |      2334 |                  16 |                    7
     5.0 |      1500 |                  10 |                    5
     6.0 |      1046 |                   7 |                    3
     8.0 |       593 |                   4 |                    2


## Exposure period is not the same as test duration

The sample-size calculation gives how long it takes to *assign* 4,668 users to
the experiment. It does not include the time those users need to *convert*.

Median time from first item view to purchase is 0.03 days, but p90 is 14.09 days
(`01_statistical_tests.ipynb`). A user assigned on the last day of the exposure
period may not purchase for another two weeks. Cutting the analysis at the end of
exposure would systematically undercount conversions among late-assigned users —
and if the intervention changes how *fast* people buy, that bias would not be
symmetric between arms.

Every user therefore gets the same 14-day conversion window from assignment,
regardless of when they entered. The analysis runs after the last-assigned user's
window closes.

**Total duration: 18 weeks** (16 exposure + 2 conversion window).

In [5]:
import math

# Final design uses MDE = 4pp, which requires 2,334 users per arm
# (see the MDE trade-off table above)
n_per_arm_final = 2334
total_users_needed = n_per_arm_final * 2

# Direct-path users reaching checkout: 4,058 over 92 days
daily_direct_users = 4058 / 92

exposure_days = total_users_needed / daily_direct_users
exposure_weeks = math.ceil(exposure_days / 7)

# Conversion window: p90 time from first item view to purchase was 14.09 days
# (see 01_statistical_tests.ipynb)
conversion_window_days = 14
window_weeks = math.ceil(conversion_window_days / 7)

total_weeks = exposure_weeks + window_weeks

print("Users per arm:", n_per_arm_final)
print("Exposure period:", exposure_weeks, "weeks")
print("Conversion window:", window_weeks, "weeks")
print("Total test duration:", total_weeks, "weeks")

Users per arm: 2334
Exposure period: 16 weeks
Conversion window: 2 weeks
Total test duration: 18 weeks


## Pre-analysis validity check: sample ratio mismatch

Before the primary metric is examined, the randomisation itself must be verified.

Users are assigned 50/50, so roughly 2,334 should land in each arm. If the observed
split deviates meaningfully — 52/48, say — something in the assignment pipeline is
broken: a logging failure, a redirect that fails on one variant, a bot filter
applying unevenly. When that happens the arms are no longer comparable, and no
result from the primary metric can be trusted regardless of how clean it looks.

**Check:** chi-square goodness-of-fit test on observed arm counts against the
expected 50/50 split.

**Threshold:** p < 0.001. Stricter than the usual 0.05, because this is a diagnostic
run on every experiment rather than a hypothesis being tested — at 0.05 you would
flag roughly one healthy test in twenty.

**Action on failure:** stop, debug the assignment pipeline, discard the data. Do not
analyse the primary metric and do not attempt to correct for the imbalance
statistically — an unexplained ratio mismatch means the cause is unknown, and an
unknown cause cannot be adjusted for.

In [6]:
from scipy.stats import chisquare

# Illustrative: what the check would look like on a healthy split
observed = [2334, 2334]
expected = [2334, 2334]

chi2, p = chisquare(f_obs=observed, f_exp=expected)
print("Chi-square:", round(chi2, 4), " p-value:", round(p, 4))

# And on a split that should fail
observed_bad = [2450, 2218]
chi2_bad, p_bad = chisquare(f_obs=observed_bad, f_exp=expected)
print("Imbalanced (2450/2218) - chi-square:", round(chi2_bad, 2), " p-value:", round(p_bad, 6))

Chi-square: 0.0  p-value: 1.0
Imbalanced (2450/2218) - chi-square: 11.53  p-value: 0.000685


## Final design and the trade-off behind it

**Recommendation: direct-path users only, MDE 4pp, 2,334 per arm, 18 weeks total
(16 weeks exposure + a 2-week conversion window).**

The faster option — all checkout users at 4pp, 7 weeks of exposure — was
considered and rejected. Cart-path users already see a review page, so showing
them another would likely do nothing. If the true effect is 4pp among direct
users and zero among cart users, the blended effect is roughly 1.7pp, which a
test powered for 4pp would probably miss. That risks an uninterpretable null:
seven weeks spent learning nothing.

The remaining alternative — all checkout users powered for the blended 2pp effect
— takes 26 weeks of exposure, longer than testing the target population directly.

18 weeks is long enough to warrant monitoring for drift, and this should be stated
rather than hidden. A mid-test check on assignment balance and traffic composition
is warranted, but the primary metric is not examined before the test concludes:
repeatedly testing an accumulating result inflates the false-positive rate well
beyond the stated 5%, and correcting for that properly requires a sequential
testing design that was not used here.

## Decisions fixed before launch

- **Significant positive result:** ship the review step; the mechanism hypothesis
  is supported.
- **Flat result:** the 14.5pp gap was mostly selection. Stop investing in the
  direct path and redirect effort to the top of the funnel, where only 22.7% of
  users ever view a product.
- **Negative result:** the review step adds friction without benefit. Revert.
- **Guardrail breach:** do not ship, regardless of the primary metric.
- **Sample ratio mismatch:** discard the data and debug the assignment pipeline.
  No result is interpreted from a test whose arms are not comparable.

Fixing these in advance is what prevents rationalising whichever result arrives.